In [2]:
# Just to check our API is working or not
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass
if not os.getenv('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY']=getpass("Enter GROQ_API_KEY: ")
if not os.getenv('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY']=getpass("Enter TAVILY_API_KEY: ")
if not os.getenv('PINECONE_API_KEY'):
    os.environ['PINECONE_API_KEY']=getpass("Enter PINECONE_API_KEY: ")

print("Groq Configured", bool(os.getenv('GROQ_API_KEY')))
print("Tavily Configured", bool(os.getenv('TAVILY_API_KEY')))
print("Pinecone Configured", bool(os.getenv('PINECONE_API_KEY')))


python-dotenv could not parse statement starting at line 1


Groq Configured True
Tavily Configured True
Pinecone Configured True


In [3]:
from langchain_community.document_loaders import WebBaseLoader

url='https://docs.langchain.com/oss/python/langgraph/agentic-rag'
loader = WebBaseLoader(
    web_path=url
)
raw_docs=loader.load()
print("Loaded documents:", len(raw_docs))
print("Source", raw_docs[0].metadata.get('source'))
# print('\nPreview:\n')
# print(raw_docs[0].page_content[:])

C:\Users\MR OSMAN\AppData\Local\Temp\ipykernel_2396\1536049523.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded documents: 1
Source https://docs.langchain.com/oss/python/langgraph/agentic-rag


In [4]:
#Split the docs into chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True
)
chunks = splitter.split_documents(raw_docs)
print("Total Chunks:", len(chunks))
print(chunks[0].page_content[:])

Total Chunks: 28
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase studiesGet hel

### **Creating Free Local Embeddings**

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   encode_kwargs={'normalize_embeddings': True})
sample_vector = embeddings.embed_query("What is Agentic RAg")
print("Embedding dimensions:", len(sample_vector))

c:\Users\MR OSMAN\Desktop\Agentic_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


Embedding dimensions: 384


In [6]:
# sample_vector

### **Creating a Vector Database (PineCone)**

In [9]:
import os
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# 1. Use hyphens instead of underscores for index name
INDEX_NAME = "industry-agentic-rag"
NAMESPACE = "langchain_agentic_rag"

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

# 2. Updated syntax to retrieve existing index names
existing_indexes = [index.name for index in pc.list_indexes()]

# 3. Dynamically retrieve embedding dimension to prevent dimension mismatch
embedding_dimension = len(embeddings.embed_query("test"))

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=embedding_dimension,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1',
        ),
    )

    while not pc.describe_index(INDEX_NAME).status['ready']:
        time.sleep(2)

print("Pinecone index is ready:", INDEX_NAME)

# 4. Store data into Pinecone
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

# 5. Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print('Pinecone Vector Store is ready and retriever is created.')

Pinecone index is ready: industry-agentic-rag
Pinecone Vector Store is ready and retriever is created.


In [10]:
#Load the existing Index
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "industry-agentic-rag"
NAMESPACE = "langchain_agentic_rag"

pc = Pinecone(os.environ['PINECONE_API_KEY'])
index = pc.Index(INDEX_NAME)

vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)
print("Existing Pinecone index loaded and retriever is ready:", INDEX_NAME)

Existing Pinecone index loaded and retriever is ready: industry-agentic-rag


### **Testing**

In [11]:
test_query = "What happened if the retrieved documents are not relevant?"

kb_docs = retriever.invoke(test_query)
for i, doc in enumerate(kb_docs, 1):
    print(f"\n --- KB RESULT {i} ---")
    print("Source", doc.metadata.get('source'))
    print(doc.page_content[:])


 --- KB RESULT 1 ---
Source https://docs.langchain.com/oss/python/langgraph/agentic-rag
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings


@lru_cache(maxsize=1)
def _get_retriever():
    vectorstore = InMemoryVectorStore.from_documents(
        documents=doc_splits,
        embedding=OpenAIEmbeddings(),
    )
    return vectorstore.as_retriever()
2Create the retriever toolCreate a retriever tool using the @tool decorator:from langchain.tools import tool


@tool
def retrieve_blog_posts(query: str) -> str:
    """Search and return information about Lilian Weng blog posts."""
    retriever = _get_retriever()
    retrieved_docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in retrieved_docs])


retriever_tool = retrieve_blog_posts
3Test the toolretriever_tool.invoke({"query": "types of reward hacking"})

​Generate a query or respond
With the retriever tool ready, start building the agent as a LangGraph g